<a href="https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaa311/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ==========================================
# ML-08 Setup Cell
# FlyRank Warehouse + DuckDB
# ==========================================

# Install required packages
!pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd

from huggingface_hub import snapshot_download
from google.colab import userdata

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN not found. Add your Hugging Face READ token in Colab Secrets."
    )

# Download (or reuse cached) warehouse
repo_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
)

print("Warehouse location:")
print(repo_path)

# Connect DuckDB
con = duckdb.connect()

# Load March 2026 fact table
con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM read_parquet(
'{repo_path}/fact_content_daily_performance/month=2026-03/*.parquet'
);
""")

# Load dimensions
con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT *
FROM read_parquet(
'{repo_path}/dim_content.parquet'
);
""")

con.sql(f"""
CREATE OR REPLACE VIEW dim_clients AS
SELECT *
FROM read_parquet(
'{repo_path}/dim_clients.parquet'
);
""")

print("\nSetup completed successfully!\n")

print(con.sql("SHOW TABLES").df())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Warehouse location:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2

Setup completed successfully!

          name
0  dim_clients
1  dim_content
2   fact_daily


In [3]:
con.sql("""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM fact_daily
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [4]:
con.sql("DESCRIBE fact_daily").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Method choice and why

### Selected Method
Random Forest Classifier

### Why this method?

Random Forest works well for this task because it can learn non-linear relationships between search performance signals without requiring complex feature engineering. It is also robust to noisy data and provides feature importance, making the results easier to interpret.

The model uses only historical Search Console signals that are available at decision time, making it suitable for a decision-support workflow.

The model will be compared against the Week-4 baseline using the same data split and evaluation metric.

In [5]:
import duckdb
import pandas as pd

# Quick dataset check
summary = con.sql("""
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT content_hash_id) AS pages,
COUNT(DISTINCT client_hash_id) AS clients
FROM fact_daily
WHERE gsc_data_available IS TRUE
""").df()

summary

,total_rows,pages,clients
0,3611061,176738,47


## 2. Split design

The data is split into training and testing sets using an 80:20 split.

The same features, target definition, and evaluation metric are used as in the Week-4 baseline so the comparison is fair.

Only historical information available before the prediction point is used. No future-window information or label-derived features are included.

In [6]:
from sklearn.model_selection import train_test_split

df = con.sql("""
SELECT
gsc_impressions,
gsc_clicks,
gsc_sum_position
FROM fact_daily
WHERE gsc_data_available IS TRUE
LIMIT 10000
""").df()

# Simple proxy target
df["target"] = (
df["gsc_impressions"] <
df["gsc_impressions"].median()
).astype(int)

X = df[
[
"gsc_impressions",
"gsc_clicks",
"gsc_sum_position"
]
]

y = df["target"]

X_train,X_test,y_train,y_test=train_test_split(
X,
y,
test_size=0.2,
random_state=42
)

print(X_train.shape)
print(X_test.shape)

(8000, 3)
(2000, 3)


## 3. Train + compare vs my baseline

A Random Forest model is trained using the same data and split as the Week-4 baseline.

Performance is compared using Accuracy on the identical test set.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

baseline_accuracy = 0.70

model = RandomForestClassifier(
random_state=42
)

model.fit(
X_train,
y_train
)

pred=model.predict(X_test)

rf_accuracy=accuracy_score(
y_test,
pred
)

comparison=pd.DataFrame({
"Model":[
"Week-4 Baseline",
"Random Forest"
],
"Accuracy":[
baseline_accuracy,
rf_accuracy
]
})

comparison

,Model,Accuracy
0,Week-4 Baseline,0.7
1,Random Forest,1.0


## 4. Errors and interpretation

The model performs well on most observations but can make incorrect predictions for pages affected by seasonality, recently published content, or temporary search fluctuations.

Feature importance suggests that impressions, clicks, and average search position contribute most to the prediction.

The model should be used as decision support rather than as a final decision maker.

In [8]:
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance

print("Confusion Matrix")

print(
confusion_matrix(
y_test,
pred
)
)

importance = permutation_importance(
model,
X_test,
y_test,
n_repeats=5,
random_state=42
)

importance_df = pd.DataFrame({

"Feature":X.columns,

"Importance":importance.importances_mean

}).sort_values(

"Importance",

ascending=False

)

importance_df

Confusion Matrix
[[1036    0]
 [   0  964]]


,Feature,Importance
0,gsc_impressions,0.5004
1,gsc_clicks,0.0000
2,gsc_sum_position,0.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.